# LSTM

Owner: _(name)_

**What this notebook is for.** Build the LSTM model, choose its settings on the validation period,
run it once on the test period, and save the predictions for part 3.

**What you do.** Run the cells in order. The settings section is where you experiment (try more
settings, keep the table). The "test once" cell is run at the very end, one time. Then write the methods
text for the report (brief, section 4: how the method works, its equations, why these settings).

**Rules everyone follows** (brief, sections 3 to 5):

- Data and split come from `src/`. Never re-download, never re-split, never shuffle.
- Learn on train, choose settings on validation, touch test once at the end.
- Each forecast uses only information available at that time (a rolling forecast).
- Keep a table of the settings you tried and their validation error.
- Time the training: the report must discuss computational cost.
- Save the test predictions with `common.save_predictions(...)`; it turns them into index points for part 3.

## How an LSTM works, in plain words (for the report's methods section)

An LSTM is a recurrent network whose memory cell is protected by three gates, each a small learned
switch between 0 and 1: the **forget gate** decides how much of the old memory to keep, the **input
gate** decides how much of today's information to write in, and the **output gate** decides how much of
the memory to show to the next layer. The gates let the network keep information for many days when
useful and drop it when not, which a vanilla RNN cannot do reliably.

### Setup: run this cell first (just run, nothing to change)

It fetches the repository (in Colab), loads the S&P 500 data and the shared settings, and prints the
three periods: train, validation, test. The first run in Colab may take a minute while it installs a
package or two.

If Colab asks for a GitHub token every time, add the token as a Colab secret named `GITHUB_TOKEN`
(key icon in the left sidebar). Step-by-step in `docs/START_HERE.md`.

In [ ]:
# ---- Setup (identical in every notebook). Just run it. ----
import sys, os, pathlib, subprocess, importlib.util

REPO_URL = "https://github.com/Makkaii7/sp500-forecasting.git"
EXTRA_PACKAGES = []      # what this notebook needs that Colab does not have by default

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 1. In Colab, fetch the repository (once per session). On your own computer this step is skipped.
if IN_COLAB and not pathlib.Path("sp500-forecasting").exists() and not pathlib.Path("src/config.py").exists():
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")             # the Colab secret, see docs/START_HERE.md
    except Exception:
        from getpass import getpass
        token = getpass("GitHub token (press Enter if the repository is public): ")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(["git", "clone", "--quiet", url], check=True)
if IN_COLAB and pathlib.Path("sp500-forecasting").exists():
    os.chdir("sp500-forecasting")

# 2. Install anything missing (usually only on the first run in Colab).
missing = [p for p in EXTRA_PACKAGES if importlib.util.find_spec(p) is None]
if missing:
    print("installing", ", ".join(missing), "... (about a minute)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# 3. Find the repository folder and load the shared code (src/config.py and src/common.py).
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config, common

# 4. The data everyone uses, and the three periods in time order.
df = common.load_prices()           # columns: Open, High, Low, Close, Volume, log_return; one row per trading day
y = common.target_series(df)        # the daily log return: what the models work on
train, val, test = common.split(y)  # dates from src/config.py
print("repository:", root)
print("data:", df.index.min().date(), "to", df.index.max().date(), "|", len(df), "trading days")
print(f"train       {train.index.min().date()} to {train.index.max().date()}  ({len(train)} days)  the model learns here")
print(f"validation  {val.index.min().date()} to {val.index.max().date()}  ({len(val)} days)  you compare settings here")
print(f"test        {test.index.min().date()} to {test.index.max().date()}  ({len(test)} days)  touched once, at the very end")
df.tail(3)

## 1. Cut the series into windows (just run)

The network sees the last W days (W = `config.WINDOW`) and predicts the next one. Before that, the values
are squeezed into the range -1 to 1 (scaling), using only the training period's smallest and largest
values, so nothing from later periods leaks in. The first validation and test windows borrow their history
from the days just before them, which were known at the time, so every forecast uses only the past.

In [ ]:
W = config.WINDOW
from sklearn.preprocessing import MinMaxScaler

def make_windows(scaled_values, prefix, w):
    full = np.concatenate([prefix, scaled_values])        # the days just before, then the period itself
    X, Y = [], []
    for i in range(len(scaled_values)):
        j = len(prefix) + i
        X.append(full[j - w:j]); Y.append(full[j])        # input: the w days before day j; target: day j
    return np.array(X)[..., None], np.array(Y)

def prepare(w):
    scaler = MinMaxScaler(feature_range=(-1, 1)).fit(train.to_numpy().reshape(-1, 1))   # fitted on train only
    sc = lambda s: scaler.transform(s.to_numpy().reshape(-1, 1)).ravel()
    tr, va, te = sc(train), sc(val), sc(test)
    X_tr, Y_tr = make_windows(tr[w:], tr[:w], w)
    X_va, Y_va = make_windows(va, tr[-w:], w)
    X_te, Y_te = make_windows(te, np.concatenate([tr, va])[-w:], w)
    return scaler, (X_tr, Y_tr), (X_va, Y_va), (X_te, Y_te)

scaler, (X_train, Y_train), (X_val, Y_val), (X_test, Y_test) = prepare(W)
print("shape = (number of windows, days per window, values per day)")
X_train.shape, X_val.shape, X_test.shape

## 2. The network, built from its settings (just run)

`build` makes a fresh network from four settings: **units** (how many memory cells the layer has),
**window** (how many days it sees), **dropout** (the share of units switched off at random during
training, which fights overfitting) and **learning rate** (the size of each training step). The last
layer is a single number with no activation, because a return can be any value, positive or negative.

In [ ]:
import keras
from keras import layers

def build(units=32, window=W, dropout=0.1, learning_rate=1e-3):
    keras.utils.set_random_seed(config.RANDOM_SEED)       # same seed, same result on the same machine
    model = keras.Sequential([
        keras.Input(shape=(window, 1)),
        layers.LSTM(units),
        layers.Dropout(dropout),
        layers.Dense(1),                  # one number out, no activation: a return can be any value
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate), loss="mean_squared_error", metrics=["mean_absolute_error"])
    return model

build().summary()

## 3. Compare settings on the validation period (just run, then add rows if you like)

Each row of `grid` is one set of settings; each row of the output table is its validation RMSE in index
points and its training time. **Early stopping** ends training when the validation error has not improved
for 5 epochs (an epoch is one pass over all the training windows) and keeps the best weights. The epoch
it stopped at is recorded too. This table goes into the report (brief, section 4: configuration table).

In [ ]:
def unscale(scaled, scaler):
    return scaler.inverse_transform(np.asarray(scaled).reshape(-1, 1)).ravel()   # back from -1..1 to a return

grid = [dict(units=16, window=10), dict(units=32, window=20), dict(units=64, window=20), dict(units=32, window=40)]
rows, fitted = [], {}
for g in grid:
    scaler_g, (Xtr, Ytr), (Xva, Yva), (Xte, Yte) = prepare(g["window"])
    model = build(units=g["units"], window=g["window"])
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    with common.Stopwatch() as sw:
        hist = model.fit(Xtr, Ytr, epochs=60, batch_size=32, validation_data=(Xva, Yva), callbacks=[stop], verbose=0)
    val_pred = common.to_price(pd.Series(unscale(model.predict(Xva, verbose=0), scaler_g), index=val.index), df)
    rows.append({**g, "epochs run": len(hist.history["loss"]), "val RMSE (index points)": common.rmse(common.actual_price(val.index, df), val_pred), "train seconds": sw.seconds})
    fitted[(g["units"], g["window"])] = (model, scaler_g, (Xte, Yte))
configs = pd.DataFrame(rows).round(3)
configs

## 4. The best settings: test once, in index points, and save (run once, at the end)

In [ ]:
best = configs.sort_values("val RMSE (index points)").iloc[0]          # the row with the lowest validation error
best = {k: (v.item() if hasattr(v, "item") else v) for k, v in best.items()}      # plain Python numbers
model, scaler_b, (Xte, Yte) = fitted[(int(best["units"]), int(best["window"]))]
test_pred = pd.Series(unscale(model.predict(Xte, verbose=0), scaler_b), index=test.index)
test_price = common.to_price(test_pred, df)
print("chosen:", best)
print("test MAE  (index points)", round(common.mae(common.actual_price(test.index, df), test_price), 2))
print("test RMSE (index points)", round(common.rmse(common.actual_price(test.index, df), test_price), 2))
common.save_predictions("lstm", df, test_pred, settings=best, train_seconds=float(best["train seconds"]))

## 5. Learning curve of the chosen model (write here)

The learning curve is the training error and the validation error plotted against the epochs. If the
training error keeps falling while the validation error rises, the model is memorising the training
period: overfitting. Describe the curve from the run above (where the validation error stopped improving,
and the gap to the training error), or re-fit the chosen settings without early stopping to see the whole
curve.

_your text_

## 6. Hand the results to part 3

In [ ]:
# Last step: get the two result files into GitHub so part 3 can read them.
# In Colab this downloads them to your computer; then on github.com open the results folder,
# click "Add file" > "Upload files", drop the two files, and commit.
# On your own computer: git add results/lstm_predictions.csv results/lstm_meta.json, then commit and push.
if IN_COLAB:
    from google.colab import files
    for name in ["lstm_predictions.csv", "lstm_meta.json"]:
        files.download(str(root / "results" / name))
else:
    print("saved:", root / "results" / "lstm_predictions.csv")
    print("saved:", root / "results" / "lstm_meta.json")
    print("now: git add results/lstm_predictions.csv results/lstm_meta.json  &&  git commit -m 'lstm: test predictions'  &&  git push")